# 05b · GSE65391 · RNA_array · scoring every sample on the first-visit modules

Reads `modules.rds`, `wgcna_input.rds`, `expression.rds`. Writes `projected_scores.rds` and the end
product `module_loadings.csv` in `data/run_artifacts/GSE65391/`.

**Why this step.** The modules and their eigengenes were fitted on 157 samples, the first visit of each
child (notebooks 04–05). The other 767 SLE samples (later visits, and the one sample with no visit number) and the 48 healthy
samples have no eigengene,
because they were not in the fit. Refitting on all samples would change each module's axis and give
children with many visits more weight. Instead, every sample is **scored with the fitted model**, the
standard way to place new observations on principal components learned from a training set
(Jolliffe IT. *Principal Component Analysis*, 2nd ed., Springer 2002, sections 3.1 and 9.2;
Sharma G et al. projectR. *Bioinformatics* 2020;36:3592–3593, doi:10.1093/bioinformatics/btaa183).

**The model.** A WGCNA module eigengene is the first principal component of the module's genes,
each standardised across the training samples (Langfelder P, Horvath S. *BMC Syst Biol* 2007;1:54,
doi:10.1186/1752-0509-1-54). `WGCNA::moduleEigengenes` computes it as the first right singular vector
v₁ of the standardised genes × samples matrix Z = U D Vᵀ, with its sign aligned to the module's average
expression. Since v₁ = Zᵀu₁ / d₁, a new sample x is scored as

  score = (z · u₁) / d₁,  with z = (x − mean) / SD, using the first-visit mean and SD of each gene.

The model for each module is therefore: the gene list, each gene's first-visit mean and SD, and the
weights u₁ / d₁ (with the sign alignment applied).

**Round-trip gate.** Scoring the 157 first visits with the model must reproduce the WGCNA eigengenes
of notebook 05: Pearson r = 1 and a maximum absolute difference at the level of rounding error.
Nothing is scored until the gate passes.

In [1]:
source("../src/paths.R")
mods <- readRDS(art("GSE65391", "modules.rds"))
w    <- readRDS(art("GSE65391", "wgcna_input.rds"))
x    <- readRDS(art("GSE65391", "expression.rds"))
fv   <- rownames(w$datExpr)
modules <- setdiff(unique(mods$colors), "grey")
c(first_visits = length(fv), modules = length(modules), all_samples = ncol(x$E))

first_visits      modules  all_samples 
         157           16          972

In [2]:
model <- lapply(setNames(modules, modules), function(mo) {
  g  <- names(mods$colors)[mods$colors == mo]
  X  <- w$datExpr[, g]                                   # first visits x module genes
  mu <- colMeans(X); s <- apply(X, 2, sd)
  Z  <- t((t(X) - mu) / s)                                # standardised, samples x genes
  sv <- svd(t(Z), nu = 1, nv = 1)                          # genes x samples = U D V'
  wt <- sv$u[, 1] / sv$d[1]
  sc <- Z %*% wt                                           # equals sv$v[, 1]
  if (cor(sc, rowMeans(Z)) < 0) wt <- -wt                  # WGCNA's alignment with average expression
  data.frame(module = mo, gene = g, mean = mu, sd = s, weight = wt, row.names = NULL)
})

**Gate.** The model applied to the 157 first visits, against the eigengenes WGCNA returned in notebook 05.

In [3]:
score <- function(E, mdl) {                               # E: genes x samples
  sapply(mdl, function(m) colSums(((E[m$gene, , drop = FALSE] - m$mean) / m$sd) * m$weight))
}
fv_scores <- score(t(w$datExpr), model)
gate <- data.frame(module = modules,
                   r = sapply(modules, function(mo) cor(fv_scores[, mo], mods$MEs[fv, paste0("ME", mo)])),
                   max_abs_diff = sapply(modules, function(mo) max(abs(fv_scores[, mo] - mods$MEs[fv, paste0("ME", mo)]))),
                   row.names = NULL)
gate
stopifnot(all(gate$r > 1 - 1e-10), all(gate$max_abs_diff < 1e-8))
cat("GATE PASSED: the model reproduces the WGCNA eigengenes of the 157 first visits\n")

module,r,max_abs_diff
<chr>,<dbl>,<dbl>
turquoise,1,1.623701e-15
pink,1,3.358425e-15
blue,1,1.096345e-15
red,1,6.383782e-16
brown,1,1.427677e-15
yellow,1,1.859624e-15
green,1,7.424616e-16
purple,1,8.326673e-16
tan,1,1.817990e-15


GATE PASSED: the model reproduces the WGCNA eigengenes of the 157 first visits


**Result.** Gate passed for all 16 modules: r = 1, largest absolute difference 3 × 10⁻¹⁵.

**Score every array sample**: 924 SLE samples (all visits) and 48 healthy samples. Every module gene
must be present in the full matrix (the same genes, the same study).

In [4]:
stopifnot(all(unlist(lapply(model, `[[`, "gene")) %in% rownames(x$E)))
all_scores <- score(x$E, model)
m <- x$meta[rownames(all_scores), ]
table(disease = m$disease, first_visit = m$first_visit)

         first_visit
disease   FALSE TRUE
  Healthy     2   46
  SLE       767  157

**Result.** All 972 array samples are scored: 157 SLE first visits, 767 other SLE samples and 48
healthy samples. `module_loadings.csv` (gene, first-visit mean and SD, weight, per module) is the
end product the other studies receive.

In [5]:
saveRDS(list(scores = all_scores, meta = m, model = model, gate = gate), art("GSE65391", "projected_scores.rds"))
write.csv(do.call(rbind, model), art("GSE65391", "module_loadings.csv"), row.names = FALSE)